In [ ]:
import os

import pandas as pd
import plotly.express as px
from tqdm.notebook import tqdm
import datetime
import plotly.graph_objects as go



In [ ]:
uri = os.getenv("NOWUM_TIMESCALE_URI")
sql = """
    SELECT *
    FROM smard_from_2018.prices
    WHERE timestamp >= '2023-12-20 23:00'
    AND timestamp <= '2024-12-31 23:00'
    ORDER BY timestamp ASC
"""

smard = pd.read_sql(sql, uri)
smard

,timestamp,commodity_id,price
0,2023-12-20 23:00:00,4169,30.16
1,2023-12-20 23:15:00,4169,30.16
2,2023-12-20 23:30:00,4169,30.16
3,2023-12-20 23:45:00,4169,30.16
4,2023-12-21 00:00:00,4169,25.08
...,...,...,...
36188,2024-12-31 22:00:00,4169,0.52
36189,2024-12-31 22:15:00,4169,0.52
36190,2024-12-31 22:30:00,4169,0.52
36191,2024-12-31 22:45:00,4169,0.52


In [3]:
px.line(smard, "timestamp", "price")

In [30]:
def get_reference_day_bnetza(date: datetime.date):

    dayofweek = date.weekday()

    # weekend case
    if dayofweek in [5, 6]:
        reference_day = date - pd.Timedelta(days=7)

    # monday
    elif dayofweek == 0:
        reference_day = date - pd.Timedelta(days=3)

    else:
        reference_day = date - pd.Timedelta(days=1)

    return reference_day

In [62]:
def select_mins_no_overlap(df: pd.DataFrame, window_h: int = 4) -> list[tuple[str]]:
    df = df.copy()

    df = df[(df["timestamp"].dt.hour >= 6) & (df["timestamp"].dt.hour <= 21)]

    df.sort_values("price", inplace=True, ignore_index=True)

    window_1_min = df.loc[0, "timestamp"]
    window_1_start = window_1_min - pd.Timedelta(hours=window_h / 2)
    window_1_end = window_1_min + pd.Timedelta(hours=window_h / 2)
    window_1 = (window_1_start, window_1_end)

    df = df[(df["timestamp"] <= window_1_start - pd.Timedelta(hours=2)) | (df["timestamp"] >= window_1_end + pd.Timedelta(hours=2))]
    df.sort_values("price", inplace=True, ignore_index=True)

    window_2_min = df.loc[0, "timestamp"]
    window_2_start = window_2_min - pd.Timedelta(hours=2, minutes=15)
    window_2_end = window_2_min + pd.Timedelta(hours=2)
    window_2 = (window_2_start, window_2_end)


    return [window_1, window_2]

In [53]:
def plot_windows(prices: pd.DataFrame, windows: dict[datetime.date, tuple]) -> go.Figure:

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=prices['timestamp'],
        y=prices['price'],
        mode='lines',
        name='Price',
        line=dict(color='blue', width=2)
    ))

    # feste Farben pro Fensterposition
    colors = [
        'rgba(0, 0, 255, 0.2)',  # erstes Fenster → blau
        'rgba(0, 255, 0, 0.2)'   # zweites Fenster → grün
    ]

    shapes = []

    for windows in windows.values():
        for i, (window_start, window_end) in enumerate(windows):
            shapes.append(dict(
                type="rect",
                x0=window_start,
                x1=window_end,
                y0=0,
                y1=1,
                xref="x",
                yref="paper",
                fillcolor=colors[i % 2],  # 0 = blau, 1 = grün
                opacity=0.8,
                layer="below",
                line_width=0
            ))

    fig.update_layout(
        shapes=shapes,
        xaxis_title='Timestamp',
        yaxis_title='Price',
        height=600,
        hovermode='x unified'
    )

    fig.show()

In [63]:
def calc_bnetza_windows(df: pd.DataFrame) -> dict[datetime.date, tuple]:
    df = df.copy()
    dates = df[df["timestamp"] >= '2023-12-31']["timestamp"].dt.date.unique()

    all_windows = {}

    for date in tqdm(dates):

        reference_day = get_reference_day_bnetza(date)
        reference_df = df[df["timestamp"].dt.date == reference_day]

        day_windows = select_mins_no_overlap(reference_df, window_h=4)
        all_windows[date] = day_windows

    return all_windows

bnetza_windows = calc_bnetza_windows(smard)
plot_windows(smard, bnetza_windows)

  0%|          | 0/367 [00:00<?, ?it/s]

In [65]:
def calc_da_windows(df: pd.DataFrame) -> dict[datetime.date, tuple]:
    df = df.copy()
    dates = df[(df["timestamp"] >= '2023-12-31') & (df["timestamp"] <= '2024-12-30')]["timestamp"].dt.date.unique()

    all_windows = {}

    for date in tqdm(dates):

        reference_day = date + pd.Timedelta(days=1)
        reference_df = df[df["timestamp"].dt.date == reference_day]

        day_windows = select_mins_no_overlap(reference_df, window_h=4)
        all_windows[date] = day_windows

    return all_windows

da_windows = calc_da_windows(smard)
plot_windows(smard, da_windows)

  0%|          | 0/366 [00:00<?, ?it/s]